# Analisi dei risultati della simulazione phishing

Notebook aggiornato allo schema `decision`, `flow_outcome`, `compromise_action`, `motivation`.

In [ ]:
from pathlib import Path
import json
import pandas as pd
import matplotlib.pyplot as plt

RESULTS_DIR = Path("results")
PLOTS_DIR = RESULTS_DIR / "plots"
MESSAGES_PATH = Path("scenarios/messages.json")
PLOTS_DIR.mkdir(parents=True, exist_ok=True)

pd.set_option("display.max_columns", None)

csv_files = sorted(p for p in RESULTS_DIR.glob("sim_*.csv") if p.stat().st_size > 0)
if not csv_files:
    raise FileNotFoundError("Nessun CSV di simulazione trovato in results/.")

csv_path = csv_files[-1]
df = pd.read_csv(csv_path)
message_config = {m["id"]: m for m in json.loads(MESSAGES_PATH.read_text(encoding="utf-8"))}

csv_path, df.shape


## Normalizzazione

Il CSV nuovo viene usato direttamente. Gli alias storici restano solo come fallback di lettura.

In [ ]:
PHISHING_TYPE = "phishing"
LEGITIMATE_TYPE = "legittimo"
DECISION_PROCEED = "PROCEDE_CON_LA_RICHIESTA"
DECISION_VERIFY = "VERIFICA_TRAMITE_CANALE_UFFICIALE"
DECISION_REPORT = "SEGNALA_COME_PHISHING"
DECISION_IGNORE = "IGNORA"
DECISION_DELAY = "RIMANDA_O_NON_DECIDE"
FLOW_STOPPED = "SI_FERMA_PRIMA_DELLA_COMPROMISSIONE"
FLOW_COMPROMISED = "COMPROMISSIONE_COMPLETATA"
FLOW_LEGITIMATE = "AZIONE_LEGITTIMA_COMPLETATA"
NO_COMPROMISE_ACTION = "NESSUNA"

def normalize_bool(series):
    if series.dtype == bool:
        return series.fillna(False)
    return series.astype(str).str.lower().isin(["true", "1", "yes", "si", "sì"])

def ensure_bool_column(name, fallback):
    if name in df.columns:
        df[name] = normalize_bool(df[name])
    else:
        df[name] = fallback

if "message_type" in df.columns:
    df["message_type"] = df["message_type"].replace({
        "legitimo": LEGITIMATE_TYPE,
        "legitimate": LEGITIMATE_TYPE,
        "legittima": LEGITIMATE_TYPE,
    })

if "decision" not in df.columns:
    df["decision"] = df.get("initial_reaction", "PARSE_ERROR")
if "flow_outcome" not in df.columns:
    df["flow_outcome"] = ""
if "compromise_action" not in df.columns:
    df["compromise_action"] = df.get("specific_action", df.get("final_action", NO_COMPROMISE_ACTION))

ensure_bool_column("parse_error", pd.Series(False, index=df.index))
ensure_bool_column("entered_flow", df["decision"].eq(DECISION_PROCEED))
ensure_bool_column("stopped_before_compromise", df["flow_outcome"].eq(FLOW_STOPPED))
ensure_bool_column("compromised", pd.Series(False, index=df.index))
ensure_bool_column("verified", df["decision"].eq(DECISION_VERIFY))
ensure_bool_column("reported", df["decision"].eq(DECISION_REPORT))
ensure_bool_column("ignored", df["decision"].eq(DECISION_IGNORE))
ensure_bool_column("delayed", df["decision"].eq(DECISION_DELAY))
ensure_bool_column("legitimate_completion", df["flow_outcome"].eq(FLOW_LEGITIMATE))

df.loc[df["flow_outcome"].eq("") & ~df["entered_flow"], "flow_outcome"] = "NON_ENTRA_NEL_FLOW"
df.loc[df["compromise_action"].isna() | df["compromise_action"].eq(""), "compromise_action"] = NO_COMPROMISE_ACTION

valid_df = df[~df["parse_error"]].copy()
phishing_df = valid_df[valid_df["message_type"] == PHISHING_TYPE].copy()
legit_df = valid_df[valid_df["message_type"] == LEGITIMATE_TYPE].copy()

summary = pd.DataFrame({
    "metrica": [
        "righe totali",
        "righe valide",
        "parse error",
        "messaggi phishing validi",
        "messaggi legittimi validi",
        "agenti unici",
        "scenari unici",
    ],
    "valore": [
        len(df),
        len(valid_df),
        int(df["parse_error"].sum()),
        len(phishing_df),
        len(legit_df),
        df["agent_id"].nunique() if "agent_id" in df.columns else None,
        df["message_id"].nunique() if "message_id" in df.columns else None,
    ],
})
summary


## Distribuzioni globali

In [ ]:
def distribution(column):
    table = valid_df[column].value_counts(dropna=False).rename_axis(column).reset_index(name="count")
    table["percentage"] = table["count"] / table["count"].sum() * 100 if not table.empty else 0
    return table

decision_distribution = distribution("decision")
flow_outcome_distribution = distribution("flow_outcome")
compromise_action_distribution = distribution("compromise_action")

decision_distribution, flow_outcome_distribution, compromise_action_distribution


In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(18, 5))
for ax, table, column, title in [
    (axes[0], decision_distribution, "decision", "Decision"),
    (axes[1], flow_outcome_distribution, "flow_outcome", "Flow outcome"),
    (axes[2], compromise_action_distribution, "compromise_action", "Compromise action"),
]:
    table.plot(kind="bar", x=column, y="percentage", legend=False, ax=ax)
    ax.set_title(title)
    ax.set_xlabel("")
    ax.set_ylabel("Percentuale")
    ax.tick_params(axis="x", rotation=45)
plt.tight_layout()
plt.savefig(PLOTS_DIR / "global_decision_flow_compromise_distribution.png", dpi=200)
plt.show()


## Metriche principali

In [ ]:
def metric(label, count, total):
    return {
        "metrica": label,
        "count": int(count),
        "totale": int(total),
        "percentuale": (count / total * 100) if total else 0,
    }

metrics = pd.DataFrame([
    metric("entered_flow rate", valid_df["entered_flow"].sum(), len(valid_df)),
    metric("stopped_before_compromise rate", valid_df["stopped_before_compromise"].sum(), len(valid_df)),
    metric("compromised rate sui phishing", phishing_df["compromised"].sum(), len(phishing_df)),
    metric("messaggi legittimi compromised", legit_df["compromised"].sum(), len(legit_df)),
    metric("legitimate_completion sui legittimi", legit_df["legitimate_completion"].sum(), len(legit_df)),
    metric("reported rate", valid_df["reported"].sum(), len(valid_df)),
    metric("verified rate", valid_df["verified"].sum(), len(valid_df)),
    metric("ignored rate", valid_df["ignored"].sum(), len(valid_df)),
    metric("delayed rate", valid_df["delayed"].sum(), len(valid_df)),
])
metrics


## Risultati per scenario e archetipo

In [ ]:
rate_columns = [
    "entered_flow",
    "stopped_before_compromise",
    "compromised",
    "reported",
    "verified",
    "ignored",
    "delayed",
    "legitimate_completion",
]

by_scenario = (
    valid_df
    .groupby(["message_id", "scenario_description", "message_type"], dropna=False)
    .agg(n=("decision", "size"), **{col: (col, "mean") for col in rate_columns})
    .reset_index()
)
by_scenario[rate_columns] = by_scenario[rate_columns] * 100
by_scenario.sort_values(["message_type", "compromised"], ascending=[True, False])


In [ ]:
def archetype_from_agent(agent_id):
    agent_id = str(agent_id)
    if "_" not in agent_id:
        return agent_id
    prefix, suffix = agent_id.rsplit("_", 1)
    return prefix if suffix.isdigit() else agent_id

valid_df["archetype"] = valid_df["agent_id"].map(archetype_from_agent)
by_archetype = (
    valid_df
    .groupby("archetype", dropna=False)
    .agg(n=("decision", "size"), **{col: (col, "mean") for col in rate_columns})
    .reset_index()
)
by_archetype[rate_columns] = by_archetype[rate_columns] * 100
by_archetype.sort_values("compromised", ascending=False)


## Warning di qualita

In [ ]:
warnings = []
forbidden_labels = ["APRE_" + "MESSAGGIO_O_LINK", "CLICCA_" + "LINK_SENZA_INSERIRE_DATI"]

if not valid_df["decision"].eq(DECISION_REPORT).any():
    warnings.append("SEGNALA_COME_PHISHING = 0")

proceed_phishing_rate = phishing_df["decision"].eq(DECISION_PROCEED).mean() * 100 if len(phishing_df) else 0
if proceed_phishing_rate > 80:
    warnings.append("PROCEDE_CON_LA_RICHIESTA > 80% sui phishing")

for message_id, group in phishing_df.groupby("message_id"):
    possible_actions = message_config.get(message_id, {}).get("possible_compromise_actions", [])
    if possible_actions and not group["flow_outcome"].eq(FLOW_COMPROMISED).any():
        warnings.append(f"COMPROMISSIONE_COMPLETATA = 0 nello scenario {message_id}")

if len(legit_df) and legit_df["compromised"].sum() > 0:
    warnings.append("Messaggi legittimi con compromised > 0")

for label in forbidden_labels:
    if df.astype(str).apply(lambda col: col.str.contains(label, regex=False, na=False)).any().any():
        warnings.append(f"Etichetta obsoleta ancora presente: {label}")

pd.DataFrame({"warning": warnings or ["nessun warning"]})


## Esportazione tabelle

In [ ]:
decision_distribution.to_csv(PLOTS_DIR / "table_decision_distribution.csv", index=False)
flow_outcome_distribution.to_csv(PLOTS_DIR / "table_flow_outcome_distribution.csv", index=False)
compromise_action_distribution.to_csv(PLOTS_DIR / "table_compromise_action_distribution.csv", index=False)
metrics.to_csv(PLOTS_DIR / "table_main_metrics.csv", index=False)
by_scenario.to_csv(PLOTS_DIR / "table_results_by_scenario.csv", index=False)
by_archetype.to_csv(PLOTS_DIR / "table_results_by_archetype.csv", index=False)
print(f"Tabelle e grafici salvati in: {PLOTS_DIR}")


## Interpretazione metodologica

I risultati sono output di una simulazione controllata, non misurazioni statistiche su persone reali. Il nuovo schema distingue la decisione iniziale dall'esito del flow: procedere significa entrare nella procedura richiesta, mentre la compromissione viene conteggiata solo quando `flow_outcome` e `compromise_action` indicano che l'azione rischiosa dello scenario e stata completata. I messaggi legittimi vanno controllati separatamente: `compromised` deve rimanere pari a zero.